# Apache Hudi - MERGE_ON_READ & Compaction

Apache Hudi supports two main table types: Copy-On-Write (COW) and Merge-On-Read (MOR). MOR tables are optimized for faster writes by storing updates in delta log files and merging them with base files during reads or compaction.

Unlike COW tables, where every update rewrites Parquet files, MOR tables can defer expensive file rewriting and compact data later.

## What you will learn

In this notebook, you will learn:

- What a Hudi Merge-On-Read (MOR) table is
- Difference between base files and delta log files
- Creating a MOR table using Spark
- Writing initial records and updates into a MOR table
- Reading MOR tables using snapshot and read-optimized queries
- Inspecting Hudi commit metadata columns
- Understanding inline compaction basics
- Checking table files and partitions on disk

## Step 1 — Create Spark session

This notebook assumes that the Hudi bundle JAR is already available in the Spark Docker image under `/opt/spark/jars`.

Because of that, we do not use `spark.jars.packages` here. This avoids Ivy/Maven downloads from inside the notebook.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-04-MOR-Compaction")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.2


## Step 2 — Define shared paths and table name

Each notebook is independent and uses its own table path. This avoids dependency on previous notebooks and makes the notebook safe to rerun after a kernel restart.

In [2]:
BASE_PATH = "/workspace/data/hudi"
TABLE_NAME = "riders_mor_compaction"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)

Table name: riders_mor_compaction
Table path: /workspace/data/hudi/riders_mor_compaction


## Step 3 — Clean previous run

For a tutorial notebook, it is useful to start from a clean table path every time.

In [3]:
import shutil
import os

if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed existing table path: {TABLE_PATH}")
else:
    print("No existing table path found.")

Removed existing table path: /workspace/data/hudi/riders_mor_compaction


## Step 4 — Prepare baseline data

We will create a small rider dataset. The `rider` column will be the Hudi record key, `city` will be the partition path, and `ts` will be the precombine field.

In [4]:
from pyspark.sql.functions import to_timestamp

df_mor_initial = spark.createDataFrame(
    [
        ("r3", "LAX", "2024-01-05 12:00:00"),
        ("r4", "SEA", "2024-01-05 12:30:00"),
    ],
    ["rider", "city", "ts"]
).withColumn("ts", to_timestamp("ts"))

df_mor_initial.show(truncate=False)
df_mor_initial.printSchema()

[Stage 1:=======================================>                   (2 + 1) / 3]

+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r3   |LAX |2024-01-05 12:00:00|
|r4   |SEA |2024-01-05 12:30:00|
+-----+----+-------------------+

root
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)



## Step 5 — Write a Hudi MERGE_ON_READ table

This creates a MOR table. MOR tables can store updates in delta log files and later compact them into base files.

For this demo, inline compaction is enabled so Hudi may compact automatically after a configured number of delta commits.

In [6]:
hudi_write_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.table.type": "MERGE_ON_READ",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.compact.inline": "true",
    "hoodie.compact.inline.max.delta.commits": "2",
}

(
    df_mor_initial.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("overwrite")
    .save(TABLE_PATH)
)

print(f"Created Hudi MOR table at: {TABLE_PATH}")

26/04/25 19:39:54 WARN HoodieSparkSqlWriterInternal: hoodie table at /workspace/data/hudi/riders_mor_compaction already exists. Deleting existing data & overwriting with new data.


[Stage 76:>                                                         (0 + 2) / 2]

26/04/25 19:40:09 WARN HoodieTableFileSystemView: Partition: SEA is not available in store
26/04/25 19:40:09 WARN HoodieTableFileSystemView: Partition: LAX is not available in store


Created Hudi MOR table at: /workspace/data/hudi/riders_mor_compaction


## Step 6 — Read the MOR table as a snapshot

A snapshot query returns the latest view of the table by merging base files and log files when needed.

In [7]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_mor")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path
FROM riders_mor
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+-------------------+------------------+----------------------+
|rider|city|ts                 |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|
+-----+----+-------------------+-------------------+------------------+----------------------+
|r3   |LAX |2024-01-05 12:00:00|20260425193955673  |r3                |LAX                   |
|r4   |SEA |2024-01-05 12:30:00|20260425193955673  |r4                |SEA                   |
+-----+----+-------------------+-------------------+------------------+----------------------+



## Step 7 — Upsert a newer version of an existing record

We update rider `r3` with a newer timestamp. Hudi uses the record key and precombine field to keep the latest version of the record.

In [8]:
df_mor_update = spark.createDataFrame(
    [
        ("r3", "LAX", "2024-01-05 13:00:00"),
    ],
    ["rider", "city", "ts"]
).withColumn("ts", to_timestamp("ts"))

df_mor_update.show(truncate=False)

(
    df_mor_update.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("append")
    .save(TABLE_PATH)
)

print("Upsert completed.")

+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r3   |LAX |2024-01-05 13:00:00|
+-----+----+-------------------+



Upsert completed.


## Step 8 — Query the latest snapshot

The snapshot query should show only the latest version of `r3`.

In [9]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_mor_latest")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time
FROM riders_mor_latest
WHERE rider = 'r3'
""").show(truncate=False)

26/04/25 19:40:56 WARN CacheManager: Asked to cache already cached data.
26/04/25 19:40:58 WARN CacheManager: Asked to cache already cached data.
+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r3   |LAX |2024-01-05 13:00:00|20260425194030639  |
+-----+----+-------------------+-------------------+



## Step 9 — Inspect commit instants from Hudi metadata columns

Instead of reading timeline files directly from `.hoodie`, this notebook reads commit information from Hudi metadata columns. This is more stable across Hudi versions.

In [10]:
latest_df = spark.read.format("hudi").load(TABLE_PATH)

commits = [
    row["_hoodie_commit_time"]
    for row in latest_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

print("Commit instants visible in the table:")
for commit in commits:
    print(commit)

latest_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "rider",
    "city",
    "ts"
).orderBy("rider").show(truncate=False)

Commit instants visible in the table:
20260425193955673
20260425194030639
+-------------------+------------------+-----+----+-------------------+
|_hoodie_commit_time|_hoodie_record_key|rider|city|ts                 |
+-------------------+------------------+-----+----+-------------------+
|20260425194030639  |r3                |r3   |LAX |2024-01-05 13:00:00|
|20260425193955673  |r4                |r4   |SEA |2024-01-05 12:30:00|
+-------------------+------------------+-----+----+-------------------+



## Step 10 — Read-optimized query

Read-optimized queries are useful for MOR tables when you want to read only compacted/base file data.

Depending on compaction timing, read-optimized results may not include the newest delta log updates until compaction has completed.

In [11]:
read_optimized_df = (
    spark.read
    .format("hudi")
    .option("hoodie.datasource.query.type", "read_optimized")
    .load(TABLE_PATH)
)

read_optimized_df.createOrReplaceTempView("riders_mor_ro")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time
FROM riders_mor_ro
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r3   |LAX |2024-01-05 13:00:00|20260425194030639  |
|r4   |SEA |2024-01-05 12:30:00|20260425193955673  |
+-----+----+-------------------+-------------------+



## Step 11 — Inspect files created by the MOR table

MOR tables may contain both base Parquet files and delta log files. The exact files depend on your Hudi version, write operations, and compaction settings.

In [12]:
from pathlib import Path

table_path = Path(TABLE_PATH)

print("Table files:")
for path in sorted(table_path.rglob("*")):
    if path.is_file():
        relative_path = path.relative_to(table_path)
        print(relative_path)

Table files:
.hoodie/.hoodie.properties.crc
.hoodie/.index_defs/.index.json.crc
.hoodie/.index_defs/index.json
.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/.hoodie.properties.crc
.hoodie/metadata/.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000_20260425193957720.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001_20260425193959576.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002_20260425194002323.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.20260

## Step 12 — Summary

In this notebook, you created an independent Hudi Merge-On-Read table and tested an upsert flow.

Key takeaways:

- MOR tables can write updates efficiently using delta logs.
- Snapshot queries return the latest merged view of the table.
- Read-optimized queries focus on compacted/base file data.
- Compaction controls when delta log data is merged into base files.
- Hudi metadata columns are useful for inspecting commit behavior.